# 4 - Checkpointing & resumable execution

Long-running agents need state. LangGraph checkpoints every node transition to a backend (SQLite for dev, Postgres for prod), and one primitive buys you four production patterns:

1. **Resume after a crash** — re-attach to the same `thread_id`, pick up where you left off.
2. **Inspect history** — every checkpoint queryable, useful for "why did the agent decide that?"
3. **Time travel** — rewind to an earlier checkpoint and rerun with a tweak.
4. **Human-in-the-loop** — pause before a node, let a human approve, resume.

We use a tiny SDR pipeline (`research → draft → review → send`) to make these visible.

In [7]:
import _path_setup  # noqa: F401
import os, time, uuid
from pathlib import Path
from typing import TypedDict
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from shared import make_checkpointer, get_llm

load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY"

CHECKPOINT_PATH = Path("data/nb4_checkpoints.sqlite")
CHECKPOINT_PATH.parent.mkdir(exist_ok=True)
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
checkpointer = make_checkpointer("dev", path=CHECKPOINT_PATH)
llm = get_llm("openai/gpt-5.4-nano")

## The pipeline

Four nodes, each just an LLM call. `crash_in_node` is the only knob — set it to a node name and that node raises. `done` tracks what ran so we can prove resume actually skipped the upstream nodes.

In [2]:
class S(TypedDict, total=False):
    prospect_name: str; company: str
    research_brief: str; draft: str; final_email: str
    sent: bool; crash_in_node: str; done: list


def _node(name, prompt_fn, output_key):
    def fn(s: S) -> S:
        if s.get("crash_in_node") == name:
            raise RuntimeError(f"injected failure in {name}")
        out = llm.invoke([HumanMessage(content=prompt_fn(s))]).content
        return {output_key: str(out), "done": (s.get("done") or []) + [name]}
    return fn


research = _node("research", lambda s: f"3-bullet research brief on {s['prospect_name']} at {s['company']}. Make plausible details up.", "research_brief")
draft    = _node("draft",    lambda s: f"4-sentence cold-outreach email using:\n{s['research_brief']}", "draft")
review   = _node("review",   lambda s: f"Tighten this email in 1 pass:\n{s['draft']}", "final_email")
def send(s: S) -> S:
    if s.get("crash_in_node") == "send": raise RuntimeError("injected failure in send")
    return {"sent": True, "done": s["done"] + ["send"]}

b = StateGraph(S)
for n, fn in [("research", research), ("draft", draft), ("review", review), ("send", send)]:
    b.add_node(n, fn)
b.add_edge(START, "research"); b.add_edge("research", "draft")
b.add_edge("draft", "review"); b.add_edge("review", "send"); b.add_edge("send", END)
graph = b.compile(checkpointer=checkpointer)

## 1. Happy path + checkpoint trail

In [3]:
tid_a = f"happy-{uuid.uuid4().hex[:6]}"
cfg_a = {"configurable": {"thread_id": tid_a}}
final = graph.invoke({"prospect_name": "Anna Schmidt", "company": "Acme", "done": []}, config=cfg_a)
print("done:", final["done"], "| sent:", final["sent"])

print(f"\n{len(list(graph.get_state_history(cfg_a)))} checkpoints persisted:")
for snap in reversed(list(graph.get_state_history(cfg_a))):
    print(f"  step={snap.metadata.get('step'):>3}  next={snap.next}  done={snap.values.get('done')}")

done: ['research', 'draft', 'review', 'send'] | sent: True

6 checkpoints persisted:
  step= -1  next=('__start__',)  done=None
  step=  0  next=('research',)  done=[]
  step=  1  next=('draft',)  done=['research']
  step=  2  next=('review',)  done=['research', 'draft']
  step=  3  next=('send',)  done=['research', 'draft', 'review']
  step=  4  next=()  done=['research', 'draft', 'review', 'send']


## 2. Crash, then resume from the same thread

Same `thread_id`. We let `review` raise, then re-invoke. Research and draft were already checkpointed, so they don't re-run.

In [4]:
tid_b = f"crash-{uuid.uuid4().hex[:6]}"
cfg_b = {"configurable": {"thread_id": tid_b}}

try:
    graph.invoke({"prospect_name": "Brian Yu", "company": "Globex",
                  "crash_in_node": "review", "done": []}, config=cfg_b)
except RuntimeError as e:
    print("crashed:", e)

snap = graph.get_state(cfg_b)
print("paused at:", snap.next, "| done so far:", snap.values["done"])

graph.update_state(cfg_b, {"crash_in_node": None})
resumed = graph.invoke(None, config=cfg_b)
print("resumed done:", resumed["done"], "(research/draft NOT re-run)")

crashed: injected failure in review
paused at: ('review',) | done so far: ['research', 'draft']
resumed done: ['research', 'draft', 'review', 'send'] (research/draft NOT re-run)


## 3. Time travel: rewind & rerun with a tweak

Grab a past checkpoint, re-invoke from there. State is restored; downstream nodes run again.

In [5]:
# Find the snapshot whose next step is review (right after draft completed).
rewind = next(s for s in graph.get_state_history(cfg_b) if s.next == ("review",))
print(f"rewinding to step {rewind.metadata.get('step')}, next={rewind.next}")

rerun = graph.invoke(None, config=rewind.config)
print("done after rerun:", rerun["done"])

rewinding to step 3, next=('review',)
done after rerun: ['research', 'draft', 'review', 'send']


## 4. Human-in-the-loop: interrupt before `send`

Same graph, compiled with `interrupt_before=["send"]`. The graph pauses, you inspect, you resume (or don't).

In [6]:
graph_hitl = b.compile(checkpointer=checkpointer, interrupt_before=["send"])
tid_c = f"hitl-{uuid.uuid4().hex[:6]}"
cfg_c = {"configurable": {"thread_id": tid_c}}

graph_hitl.invoke({"prospect_name": "Carol Diaz", "company": "Initech", "done": []}, config=cfg_c)

snap = graph_hitl.get_state(cfg_c)
print("paused, awaiting human. next:", snap.next)
print("\n--- draft for review ---\n", snap.values["final_email"])

# In a real app: surface to UI, wait for click. Here we just approve.
final = graph_hitl.invoke(None, config=cfg_c)
print("\nsent:", final["sent"])

paused, awaiting human. next: ('send',)

--- draft for review ---
 Hi Carol,  

I’m reaching out because your work at Initech on improving order-to-cash data integrity and accelerating reconciliation—especially the ~40% reduction—really stands out. I saw you led migration/optimization of legacy reporting pipelines and introduced automated reconciliation/alerting, plus a role-based access model to strengthen auditability.  

We help teams standardize metric definitions across finance and operations and catch data issues earlier with automated controls and reconciliation workflows. Would you be open to a quick 15-minute call to compare notes on what’s driving month-end discrepancies and what it would take to scale those improvements to new business units?

sent: True


## Prod: same code, Postgres backend

```python
checkpointer = make_checkpointer("prod")  # uses DATABASE_URL
graph = b.compile(checkpointer=checkpointer)
```

The agent code doesn't know which backend it's writing to. The SDR app in Segment 6 wires exactly this in; flipping `DATABASE_URL` swaps SQLite for Postgres without touching agent code. (Cloud deploy is covered in Week 3.)